**Aluno:** Otávio Augusto Reis Nascimento

**Projeto:** SalesInsight

**Professor:** Lucas Ribeiro de Lima

**Criando um dataset fictício com gerador - Opção A**

O gerador abaixo cria propositalmente dados “sujos”, que servirão de matéria-prima para o requisito de limpeza.

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza ---
        if random.random() < 0.05:
            quantidade = None                    # valor nulo
        if random.random() < 0.04:
            preco = None                         # valor nulo
        if random.random() < 0.06:
            produto = "  " + produto + " "       # espacos extras
        if random.random() < 0.03:
            data_txt = "DATA INVALIDA"           # data invalida
        if random.random() < 0.10:
            cliente = random.choice([            # ruido no nome
                cliente.upper().replace("_", "-"),
                cliente + "!!",
                "  " + cliente,
                cliente.replace("Cliente_", "cliente#"),
            ])

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())

Dataset gerado com 200 registros.
   id_venda  data_venda      cliente   produto     categoria        regiao  \
0         1  2025-05-21  cliente#016     Mouse   Perifericos       Sudeste   
1         2  2025-09-16  Cliente_039  Notebook  Computadores         Norte   
2         3  2025-03-23  Cliente_045    Tablet     Celulares  Centro-Oeste   
3         4  2025-11-06  Cliente_017  Notebook  Computadores         Norte   
4         5  2025-07-05  Cliente_037    Tablet     Celulares           Sul   

   quantidade  preco_unitario  
0         2.0          102.90  
1         NaN         3204.57  
2         1.0         1939.76  
3         6.0         3864.87  
4        10.0         2008.14  


**1 - Inspecionando os dados**

**Objetivo:** Com o dataset criado, o primeiro passo é realizar a inspeção dos dados, nessa etapa irei utilizar o .shape e verificar as colunas, os tipos de dados, os valores nulos  e os primeiros registros.

In [2]:
print(f"Shape do DataFrame: {df_bruto.shape}")

Shape do DataFrame: (200, 8)


**▶Foi contabilizado 200 linhas e 8 colunas.**

In [3]:
print("Informações sobre as colunas e tipos de dados:")
display(df_bruto.info())

Informações sobre as colunas e tipos de dados:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_venda        200 non-null    int64  
 1   data_venda      200 non-null    object 
 2   cliente         200 non-null    object 
 3   produto         200 non-null    object 
 4   categoria       200 non-null    object 
 5   regiao          200 non-null    object 
 6   quantidade      190 non-null    float64
 7   preco_unitario  196 non-null    float64
dtypes: float64(2), int64(1), object(5)
memory usage: 12.6+ KB


None

**▶Tipo de dados encontrados: float, int e object**

In [4]:
print("Contagem de valores nulos por coluna:")
display(df_bruto.isnull().sum())

Contagem de valores nulos por coluna:


,0
id_venda,0
data_venda,0
cliente,0
produto,0
categoria,0
regiao,0
quantidade,10
preco_unitario,4


**▶Total de 14 valores nulos, sendo 10 valores nulos na coluna quantidade e 4 na coluna preco_unitario**

In [5]:
print("Primeiros 5 registros do DataFrame (para visualizar a 'sujeira'):")
display(df_bruto.head())

Primeiros 5 registros do DataFrame (para visualizar a 'sujeira'):


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,cliente#016,Mouse,Perifericos,Sudeste,2.0,102.90
1,2,2025-09-16,Cliente_039,Notebook,Computadores,Norte,NaN,3204.57
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1.0,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6.0,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10.0,2008.14


**2 - Limpeza e Tratamento dos Dados**

**Objetivo:** Realizar a limpeza e padronização do dataset, executando cada etapa em células separadas para maior controle e observação do processo. As contagens de registros removidos serão atualizadas em cada passo para o relatório final.

**Primeiro irei importar as bibliotecas, realizar a cópia do DF (como orientado em aula, para não trabalhar no DF original)**

In [6]:
import pandas as pd
import numpy as np
import re

# Criar uma cópia do DataFrame bruto para realizar a limpeza passo a passo
df_limpo = df_bruto.copy()

registros_iniciais = len(df_limpo)
removidos_data = 0
removidos_nulos_criticos = 0
clientes_nao_padronizados_nome = 0

print(f"DataFrame 'df_limpo' criado como cópia de 'df_bruto'. Total inicial de registros: {registros_iniciais}")

DataFrame 'df_limpo' criado como cópia de 'df_bruto'. Total inicial de registros: 200


### 2.1 - Remover espaços extras em colunas de texto

In [7]:
for col in ['cliente', 'produto', 'categoria', 'regiao']:
    if col in df_limpo.columns and df_limpo[col].dtype == 'object':
        df_limpo[col] = df_limpo[col].astype(str).str.strip()
print("  Espaços extras removidos.")

  Espaços extras removidos.


### 2.2 - Converter `data_venda` para datetime e descartar inválidas

In [8]:
pre_date_removal_len = len(df_limpo)
df_limpo['data_venda'] = pd.to_datetime(df_limpo['data_venda'], errors='coerce')
df_limpo.dropna(subset=['data_venda'], inplace=True)
removidos_data = pre_date_removal_len - len(df_limpo)
print(f"  Foram descartados {removidos_data} registros com datas inválidas.")
print(f"  Registros restantes: {len(df_limpo)}")

  Foram descartados 4 registros com datas inválidas.
  Registros restantes: 196


### 2.3 - Remover nulos em `quantidade` e `preco_unitario`

In [9]:
pre_null_removal_len = len(df_limpo)
df_limpo.dropna(subset=['quantidade', 'preco_unitario'], inplace=True)
removidos_nulos_criticos = pre_null_removal_len - len(df_limpo)
print(f"  Foram descartados {removidos_nulos_criticos} registros com nulos em 'quantidade' ou 'preco_unitario'.")
print(f"  Registros restantes: {len(df_limpo)}")

  Foram descartados 13 registros com nulos em 'quantidade' ou 'preco_unitario'.
  Registros restantes: 183


### 2.4 - Ajustar tipos numéricos

In [10]:
df_limpo['quantidade'] = df_limpo['quantidade'].astype(int)
df_limpo['preco_unitario'] = df_limpo['preco_unitario'].astype(float)
print("  Tipos numéricos ajustados.")
display(df_limpo[['quantidade', 'preco_unitario']].dtypes)

  Tipos numéricos ajustados.


,0
quantidade,int64
preco_unitario,float64


### 2.5 - Padronizar nomes de clientes com Regex

In [11]:
def padronizar_cliente_nome(nome_original):
    nome_limpo = re.sub(r'[^a-zA-Z0-9_]', '', str(nome_original)).strip()
    match = re.search(r'cliente_?(\d+)', nome_limpo, re.IGNORECASE)
    if match:
        return f"Cliente_{int(match.group(1)):03d}"
    return f"NAO_PADRAO_{nome_limpo}"

df_limpo['cliente'] = df_limpo['cliente'].apply(padronizar_cliente_nome)
clientes_nao_padronizados_nome = df_limpo[df_limpo['cliente'].str.startswith('NAO_PADRAO_')].shape[0]
if clientes_nao_padronizados_nome > 0:
    print(f"    Atenção: {clientes_nao_padronizados_nome} registros de clientes não puderam ser padronizados para o formato 'Cliente_NNN'.")
print("  Nomes de clientes foram padronizados.")

  Nomes de clientes foram padronizados.


### 2.6 - Relatório Final da Limpeza

In [17]:
registros_finais = len(df_limpo)
total_removidos = registros_iniciais - registros_finais

relatorio_limpeza = {
    'registros_iniciais': registros_iniciais,
    'removidos_data_invalida': removidos_data,
    'removidos_nulos_quantidade_preco': removidos_nulos_criticos,
    'clientes_nao_padronizados_nome': clientes_nao_padronizados_nome,
    'total_registros_removidos': total_removidos,
    'registros_finais': registros_finais
}

print("\n--- Relatório Final de Limpeza ---")
for key, value in relatorio_limpeza.items():
    print(f"{key.replace('_', ' ').capitalize()}: {value}")

print("\n--- Motivo da Remoção dos Registros---")
print(f"Durante o processo de limpeza, foram removidos {removidos_data} registros devido a **datas inválidas** na coluna `data_venda`. \nEstes registros continham valores que não puderam ser convertidos para o formato de data/hora válido, tornando-os inconsistentes para análise temporal.\n")
print(f"Além disso, {removidos_nulos_criticos} registros foram descartados devido à presença de **valores nulos** nas colunas `quantidade` ou `preco_unitario`. \nA ausência de dados nessas colunas é crítica, pois impede o cálculo preciso de vendas e outras métricas financeiras. \nManter registros com esses valores nulos compromete a integridade de qualquer análise futura.")

print("\n--- Informações Finais do DataFrame Limpo ---")
print("\nShape do DataFrame limpo:")
display(df_limpo.shape)

print("\nInformações sobre as colunas e tipos de dados do DataFrame limpo:")
display(df_limpo.info())

print("\nContagem de valores nulos por coluna no DataFrame limpo:")
display(df_limpo.isnull().sum())

print("\nPrimeiros 5 registros do DataFrame limpo:")
display(df_limpo.head())


--- Relatório Final de Limpeza ---
Registros iniciais: 200
Removidos data invalida: 4
Removidos nulos quantidade preco: 13
Clientes nao padronizados nome: 0
Total registros removidos: 17
Registros finais: 183

--- Motivo da Remoção dos Registros---
Durante o processo de limpeza, foram removidos 4 registros devido a **datas inválidas** na coluna `data_venda`. 
Estes registros continham valores que não puderam ser convertidos para o formato de data/hora válido, tornando-os inconsistentes para análise temporal.

Além disso, 13 registros foram descartados devido à presença de **valores nulos** nas colunas `quantidade` ou `preco_unitario`. 
A ausência de dados nessas colunas é crítica, pois impede o cálculo preciso de vendas e outras métricas financeiras. 
Manter registros com esses valores nulos compromete a integridade de qualquer análise futura.

--- Informações Finais do DataFrame Limpo ---

Shape do DataFrame limpo:


(183, 8)


Informações sobre as colunas e tipos de dados do DataFrame limpo:
<class 'pandas.core.frame.DataFrame'>
Index: 183 entries, 0 to 199
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_venda        183 non-null    int64         
 1   data_venda      183 non-null    datetime64[ns]
 2   cliente         183 non-null    object        
 3   produto         183 non-null    object        
 4   categoria       183 non-null    object        
 5   regiao          183 non-null    object        
 6   quantidade      183 non-null    int64         
 7   preco_unitario  183 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 12.9+ KB


None


Contagem de valores nulos por coluna no DataFrame limpo:


,0
id_venda,0
data_venda,0
cliente,0
produto,0
categoria,0
regiao,0
quantidade,0
preco_unitario,0



Primeiros 5 registros do DataFrame limpo:


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2025-05-21,Cliente_016,Mouse,Perifericos,Sudeste,2,102.90
2,3,2025-03-23,Cliente_045,Tablet,Celulares,Centro-Oeste,1,1939.76
3,4,2025-11-06,Cliente_017,Notebook,Computadores,Norte,6,3864.87
4,5,2025-07-05,Cliente_037,Tablet,Celulares,Sul,10,2008.14
5,6,2025-08-21,Cliente_041,Headset,Perifericos,Sudeste,2,337.41


**3 - Criação de Colunas Derivadas**

**Objetivo:** Gerar novas colunas no DataFrame `df_limpo` para enriquecer a análise, com base nas transformações condicionais e extração de componentes de data.

**3.1 - Calculando e criando a coluna de 'receita_total' usando do df_limpo**

In [18]:
df_limpo['receita_total'] = df_limpo['quantidade'] * df_limpo['preco_unitario']
print("Coluna 'receita_total' criada.")

Coluna 'receita_total' criada.


Função para verificar os primeiros registros com a nova coluna

In [19]:
display(df_limpo[['quantidade', 'preco_unitario', 'receita_total']].head())

,quantidade,preco_unitario,receita_total
0,2,102.90,205.80
2,1,1939.76,1939.76
3,6,3864.87,23189.22
4,10,2008.14,20081.40
5,2,337.41,674.82


**3.2 - Função para extrair 'mes' e 'ano'**

In [20]:
df_limpo['mes'] = df_limpo['data_venda'].dt.month
df_limpo['ano'] = df_limpo['data_venda'].dt.year
print("Colunas 'mes' e 'ano' extraídas da 'data_venda'.")

Colunas 'mes' e 'ano' extraídas da 'data_venda'.


Função para exibir os primeiros registros com as novas colunas

In [21]:
display(df_limpo[['data_venda', 'mes', 'ano']].head())

,data_venda,mes,ano
0,2025-05-21,5,2025
2,2025-03-23,3,2025
3,2025-11-06,11,2025
4,2025-07-05,7,2025
5,2025-08-21,8,2025


**3.3 - Criação da coluna mes_nome usando um dicionário de mapeamento conforme recomendado no na atividade.**

In [22]:
mes_map = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
df_limpo['mes_nome'] = df_limpo['mes'].map(mes_map)
print("Coluna 'mes_nome' criada.")

Coluna 'mes_nome' criada.


Função para exibir os primeiros registros com a nova coluna

In [23]:
display(df_limpo[['mes', 'mes_nome']].head())

,mes,mes_nome
0,5,Maio
2,3,Março
3,11,Novembro
4,7,Julho
5,8,Agosto


**3.4 - Criando a função 'trimestre'**

In [24]:
df_limpo['trimestre'] = 'Q' + df_limpo['data_venda'].dt.quarter.astype(str)
print("Coluna 'trimestre' criada.")

Coluna 'trimestre' criada.


Função para exibir os primeiros registros com a nova coluna

In [25]:
display(df_limpo[['data_venda', 'trimestre']].head())

,data_venda,trimestre
0,2025-05-21,Q2
2,2025-03-23,Q1
3,2025-11-06,Q4
4,2025-07-05,Q3
5,2025-08-21,Q3


**3.5 - Transformação condicional vetorizada**
Função para criar 'faixa_receita_item' com classificação condicional.

In [26]:
condicoes = [
    df_limpo["receita_total"] < 500,
    (df_limpo["receita_total"] >= 500) & (df_limpo["receita_total"] < 5000),
    df_limpo["receita_total"] >= 5000,
]
faixas = ["Baixo Valor", "Medio Valor", "Alto Valor"]
df_limpo["faixa_receita_item"] = np.select(condicoes, faixas, default="Nao Classificado")
print("Coluna 'faixa_receita_item' criada.")

Coluna 'faixa_receita_item' criada.


Função para exibir os primeiros registros com a nova coluna

In [27]:
display(df_limpo[['receita_total', 'faixa_receita_item']].head())

,receita_total,faixa_receita_item
0,205.80,Baixo Valor
2,1939.76,Medio Valor
3,23189.22,Alto Valor
4,20081.40,Alto Valor
5,674.82,Medio Valor


**4 - Métricas Agregadas**

**Objetivo:** Calcular e exibir métricas importantes utilizando agrupamentos (`groupby`) para obter insights sobre vendas por diferentes dimensões. As métricas foram calculadas de acordo com Assinatura Suugerida no documento da atividade proposta.

In [31]:
def calcular_metricas(df):
    metricas = {}

    # 4.1 Receita total, quantidade vendida e número de vendas por mês
    # Usando 'mes' e 'ano' para ordenar corretamente e 'mes_nome' para exibição
    metricas['por_mes'] = df.groupby(['ano', 'mes_nome']).agg(
        receita_total=('receita_total', 'sum'),
        quantidade_vendida=('quantidade', 'sum'),
        numero_vendas=('id_venda', 'count')
    ).reset_index().sort_values(by=['ano', 'mes_nome'])

    # 4.2 Receita total por produto (Top 5, em ordem decrescente)
    metricas['top_produtos'] = df.groupby('produto').agg(
        receita_total=('receita_total', 'sum')
    ).reset_index().sort_values(by='receita_total', ascending=False).head(5)

    # 4.3 Receita total por categoria
    metricas['por_categoria'] = df.groupby('categoria').agg(
        receita_total=('receita_total', 'sum')
    ).reset_index().sort_values(by='receita_total', ascending=False)

    # 4.4 Receita total e ticket médio por região
    metricas['por_regiao'] = df.groupby('regiao').agg(
        receita_total=('receita_total', 'sum'),
        # Ticket médio é a receita total dividida pelo número de vendas na região
        ticket_medio=('receita_total', 'mean')
    ).reset_index().sort_values(by='receita_total', ascending=False)

    return metricas

# Calcular as métricas uma vez
metricas_agregadas = calcular_metricas(df_limpo)
print("Métricas agregadas calculadas e armazenadas em 'metricas_agregadas'.")

Métricas agregadas calculadas e armazenadas em 'metricas_agregadas'.


### Visualização Individual das Métricas Agregadas

Agora que todas as métricas foram calculadas pela função `calcular_metricas` e estão armazenadas no dicionário `metricas_agregadas`, podemos visualizá-las individualmente em células separadas.

In [32]:
print("\n--- Métrica: Receita, Quantidade e Número de Vendas por Mês ---")
display(metricas_agregadas['por_mes'])


--- Métrica: Receita, Quantidade e Número de Vendas por Mês ---


,ano,mes_nome,receita_total,quantidade_vendida,numero_vendas
0,2025,Abril,80636.64,48,7
1,2025,Agosto,85790.38,72,12
2,2025,Dezembro,104760.44,71,14
3,2025,Fevereiro,73895.74,70,12
4,2025,Janeiro,120866.25,92,15
5,2025,Julho,106667.30,84,14
6,2025,Junho,106534.48,95,15
7,2025,Maio,132080.62,104,19
8,2025,Março,123869.23,98,18
9,2025,Novembro,160297.77,136,23


In [33]:
print("\n--- Métrica: Top 5 Produtos por Receita ---")
display(metricas_agregadas['top_produtos'])


--- Métrica: Top 5 Produtos por Receita ---


,produto,receita_total
3,Notebook,374174.85
5,Tablet,335335.83
4,Smartphone,303255.94
1,Monitor,169554.67
0,Headset,48368.37


In [34]:
print("\n--- Métrica: Receita Total por Categoria ---")
display(metricas_agregadas['por_categoria'])


--- Métrica: Receita Total por Categoria ---


,categoria,receita_total
0,Celulares,638591.77
1,Computadores,543729.52
2,Perifericos,108025.01


In [35]:
print("\n--- Métrica: Receita Total e Ticket Médio por Região ---")
display(metricas_agregadas['por_regiao'])


--- Métrica: Receita Total e Ticket Médio por Região ---


,regiao,receita_total,ticket_medio
1,Nordeste,366321.23,8721.934048
2,Norte,299917.37,7140.889762
0,Centro-Oeste,244586.97,5688.069070
4,Sul,228249.66,6521.418857
3,Sudeste,151271.07,7203.384286
